In [1]:
import sys
from pathlib import Path
import torch
import librosa

sys.path.insert(0, str(Path.cwd()))

from acoustic.utils.config import load_config
from acoustic.models.whisper.model import load_checkpoint

In [2]:
cfg = load_config("acoustic/configs/base_config.yaml")

output_dir = cfg['training']['output_dir']
checkpoint_dir = Path(output_dir) / "final_model"

In [3]:
model, processor = load_checkpoint(str(checkpoint_dir), cfg)
#model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [4]:
audio_path = "acoustic/data/sound.wav"

audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)

inputs = processor.feature_extractor(
    audio_array,
    sampling_rate=16000,
    return_tensors="pt"
)
input_features = inputs.input_features.to(device)

In [5]:
with torch.no_grad():
    predicted_ids = model.generate(input_features)

transcription = processor.decode(predicted_ids[0], skip_special_tokens=True)

print("Распознанный текст:", transcription)

/home/abonentvneseti/programming/github/STT_russian_lang/.venv/lib/python3.11/site-packages/transformers/models/whisper/modeling_whisper.py:697: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at ../aten/src/ATen/native/transformers/hip/sdp_utils.cpp:505.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Распознанный текст:  Допустим, она должна работать.
